In [15]:
import cv2
import numpy as np
from PIL import Image
import io
import torch
from torchvision import transforms

class Ours3:
    def __init__(self, size=224, crop=0, target_bytes=65536, bit_error_rate=0.001):
        self.size = size
        self.crop = crop
        self.target_bytes = target_bytes  # 2^16
        self.bit_error_rate = bit_error_rate

    def __call__(self, img):
        # Step 1: PIL → NumPy → crop
        img_np = np.array(img.convert("RGB"))
        if self.crop > 0:
            h, w, _ = img_np.shape
            img_np = img_np[self.crop:h - self.crop, self.crop:w - self.crop, :]

        # Step 2: JPEG 압축
        jpeg_data = self.compress_to_target_size(img_np, self.target_bytes)

        # Step 3: 바이트 단위 채널 노이즈 추가
        for i in range(len(jpeg_data)):
            if np.random.rand() < self.bit_error_rate:
                jpeg_data[i] ^= np.random.randint(1, 256)

        # Step 4: 디노이징 
        byte_array = np.array(jpeg_data, dtype=np.uint8)
        kernel = np.ones(3, dtype=np.uint8) / 3
        denoised_bytes = np.convolve(byte_array, kernel, mode='same').astype(np.uint8)

        # Step 5
        try:
            restored_img = Image.open(io.BytesIO(denoised_bytes.tobytes())).convert("RGB")
        except:
            restored_img = Image.fromarray(img_np)  # fallback

        # Step 6: Resize + Normalize
        resized = restored_img.resize((self.size, self.size))
        tensor = transforms.ToTensor()(resized)
        norm_tensor = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                           std =[0.229, 0.224, 0.225])(tensor)
        return norm_tensor

    def compress_to_target_size(self, img_np, target_size):
        low, high = 1, 95  # 품질 범위
        best_data = None

        while low <= high:
            mid = (low + high) // 2
            buffer = io.BytesIO()
            Image.fromarray(img_np).save(buffer, format='JPEG', quality=mid)
            data = bytearray(buffer.getvalue())

            if len(data) > target_size:
                high = mid - 1
            else:
                best_data = data
                low = mid + 1

        if best_data is not None and len(best_data) < target_size:
            best_data += bytearray([0] * (target_size - len(best_data)))
        return best_data if best_data else bytearray(buffer.getvalue())

In [16]:
from torchvision import datasets, transforms
import torch

# JPEG + 채널 노이즈 + 디노이징 기반 전처리 클래스 사용
jpeg_transform = transforms.Compose([
    Ours3(
        size=224,              # 최종 CNN 입력 사이즈
        crop=2,                # 선택적 crop
        target_bytes=65536,   # 2^16
        bit_error_rate=0.001   # 채널 노이즈 비율 (바이트 단위)
    )
])

# 데이터셋 경로
train_path = '/home/dh/venv/dataset/Animals/Train'
test_path  = '/home/dh/venv/dataset/Animals/Test'

# 데이터셋 생성
trainset = datasets.ImageFolder(train_path, transform=jpeg_transform)
testset  = datasets.ImageFolder(test_path,  transform=jpeg_transform)

# DataLoader 정의
trainloader = torch.utils.data.DataLoader(trainset, batch_size=40, shuffle=True, num_workers=2, drop_last=True)
testloader  = torch.utils.data.DataLoader(testset,  batch_size=40, shuffle=False, num_workers=2, drop_last=True)

In [17]:
print(f"# of trainset = {len(trainset)}")
print(f"# of trainloader = {len(trainloader)}")
target, label = trainset[5000]
print(f"input size: {target.shape}, {label}")     

# of trainset = 8000
# of trainloader = 200
input size: torch.Size([3, 224, 224]), 2


In [18]:
import os
import numpy as np 
import torch.optim as optim   
from tqdm import tqdm 

def train(model, device, trainloader, optimizer, criterion, num_epochs, save_path='./prob2_3_ours3_weight1'):
    os.makedirs(save_path, exist_ok=True)  
    history = []

    for epoch in tqdm(range(num_epochs)):
        model.train()
        epoch_loss, correct, total = 0, 0, 0

        for X, y in trainloader: 
            X = X.to(device);y = y.to(device)

            optimizer.zero_grad()
            predict = model(X)
            loss = criterion(predict, y)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            pred_class = predict.argmax(dim=1)
            correct += (pred_class == y).sum().item()
            total += y.size(0)

        avg_loss = epoch_loss / len(trainloader)
        avg_accuracy = correct / total
        history.append((epoch + 1, avg_loss, avg_accuracy))
        print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {avg_loss:.6f}, Accuracy: {avg_accuracy:.4f}")

        if (epoch + 1) % 10 == 0:
            filename = os.path.join(save_path, f"weight{epoch+1}.pth")
            torch.save(model.state_dict(), filename)
            print(f"Saved checkpoint: {filename}")

    return np.array(history)
    
def test(model, device, test_loader, criterion):
    test_loss = []
    test_accuracy = []
    model.eval()

    with torch.no_grad():
        for X, y in tqdm(test_loader): 
            X = X.to(device)
            y = y.to(device)

            predict = model(X)
            loss = criterion(predict, y)

            pred_class = predict.argmax(dim=1)
            accuracy = (pred_class == y).float().mean()

            test_accuracy.append(accuracy.item())
            test_loss.append(loss.item())

    avg_loss = sum(test_loss) / len(test_loss)
    avg_accuracy = sum(test_accuracy) / len(test_accuracy)
    print(f'test loss : {avg_loss:.4f} / test_accuracy : {avg_accuracy:.4f}')

In [19]:
## resnet model ##
import torchvision.models as models
import torch
import torch.nn as nn
device = 'cuda' if torch.cuda.is_available() else 'cpu'
ours3 = models.resnet18()
num_ftrs = ours3.fc.in_features
ours3.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
ours3.fc = nn.Sequential(
    nn.Linear(num_ftrs, 4),
)
ours3 = ours3.to(device)

In [20]:
# 하이퍼파라미터 설정
num_epochs = 50
lr = 0.001
optimizer = torch.optim.Adam(ours3.parameters(), lr=lr)
criterion = torch.nn.CrossEntropyLoss()

# 학습 실행
history = train(ours3, device, trainloader, optimizer, criterion, num_epochs)

  2%|▊                                        | 1/50 [02:46<2:15:55, 166.43s/it]

Epoch [1/50] - Loss: 0.788382, Accuracy: 0.6705


  4%|█▋                                       | 2/50 [05:33<2:13:20, 166.67s/it]

Epoch [2/50] - Loss: 0.628096, Accuracy: 0.7485


  6%|██▍                                      | 3/50 [08:20<2:10:52, 167.08s/it]

...

Epoch [11/50] - Loss: 0.229380, Accuracy: 0.9170


 24%|█████████▌                              | 12/50 [33:30<1:45:55, 167.24s/it]

Epoch [12/50] - Loss: 0.178485, Accuracy: 0.9336


 24%|█████████▌                              | 12/50 [34:22<1:48:52, 171.90s/it]


KeyboardInterrupt: 

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import os
model_paths = [
    './prob2_3_ours3_weight2/weight10.pth',
    './prob2_3_ours3_weight2/weight20.pth',
    './prob2_3_ours3_weight2/weight30.pth',
    './prob2_3_ours3_weight2/weight40.pth',
    './prob2_3_ours3_weight2/weight50.pth',
]
criterion = torch.nn.CrossEntropyLoss()
device = 'cuda' if torch.cuda.is_available() else 'cpu'

for path in model_paths:
    model = models.resnet18()
    model.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
    model.fc = nn.Sequential(
        nn.Linear(model.fc.in_features, 4)
    )
    model.load_state_dict(torch.load(path, map_location=device))
    model = model.to(device)
    test(model, device, testloader, criterion)